# California House Price Prediction - Complete Pipeline

## End-to-End Machine Learning Project

This notebook demonstrates the complete pipeline from data loading to model evaluation.

In [ ]:
# Import libraries
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
from data_loader import load_california_housing_data, get_data_summary
from preprocessing import preprocess_pipeline
from feature_engineering import engineer_features
from models import HousePriceModels
from evaluation import evaluate_all_models, plot_model_comparison, plot_predictions, plot_residuals_analysis
from utils import save_model, print_metrics

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All libraries imported successfully!")

## 1. Data Loading

In [ ]:
# Load data
df, feature_descriptions = load_california_housing_data(save_to_csv=True)

print("\n📊 Dataset Shape:", df.shape)
print("\n📋 First few rows:")
df.head()

In [ ]:
# Data summary
summary = get_data_summary(df)
print("\n📈 Statistical Summary:")
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
# Distribution plots
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.ravel()

for idx, col in enumerate(df.columns):
    axes[idx].hist(df[col], bins=50, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{col} Distribution', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n🔗 Correlation with Target (MedHouseVal):")
print(correlation_matrix['MedHouseVal'].sort_values(ascending=False))

In [ ]:
# Geographic visualization
plt.figure(figsize=(14, 10))
scatter = plt.scatter(df['Longitude'], df['Latitude'], 
                     c=df['MedHouseVal'], cmap='viridis', 
                     alpha=0.4, s=df['Population']/10, edgecolors='k', linewidth=0.5)
plt.colorbar(scatter, label='Median House Value ($100k)')
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.title('California Housing Prices - Geographic Distribution', fontsize=16, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# Engineer features
feature_results = engineer_features(df, use_clusters=True, use_distance=True, use_polynomial=False)
df_engineered = feature_results['engineered_df']

print("\n✨ Feature Engineering Complete!")
print(f"Original features: {len(df.columns)}")
print(f"Engineered features: {len(df_engineered.columns)}")
print(f"\n📋 New features: {[col for col in df_engineered.columns if col not in df.columns]}")

## 4. Data Preprocessing

In [ ]:
# Preprocess data
processed_data = preprocess_pipeline(df_engineered, outlier_method='clip', scale_method='robust')

X_train = processed_data['X_train_scaled']
X_test = processed_data['X_test_scaled']
y_train = processed_data['y_train']
y_test = processed_data['y_test']

print("\n✅ Preprocessing Complete!")

## 5. Model Training

In [ ]:
# Initialize and train models
model_manager = HousePriceModels()
model_manager.initialize_models()

# Train all models
training_results = model_manager.train_all_models(X_train, y_train, cv=5)

In [ ]:
# Hyperparameter tuning for top models
print("\n🔧 Hyperparameter Tuning...\n")

for model_name in ['Random Forest', 'XGBoost', 'LightGBM']:
    tuning_result = model_manager.hyperparameter_tuning(model_name, X_train, y_train, method='grid', cv=3)
    print(f"\n{model_name} tuning complete!")

In [ ]:
# Create ensemble models
voting_ensemble = model_manager.create_ensemble(X_train, y_train, ensemble_type='voting')
stacking_ensemble = model_manager.create_ensemble(X_train, y_train, ensemble_type='stacking')

## 6. Model Evaluation

In [ ]:
# Evaluate all models
results_df, predictions = evaluate_all_models(model_manager.trained_models, X_test, y_test)

print("\n" + "="*80)
print("MODEL PERFORMANCE COMPARISON")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

In [ ]:
# Plot model comparison
plot_model_comparison(results_df)

In [ ]:
# Plot predictions for top 3 models
plot_predictions(y_test, predictions, top_n=3)

In [ ]:
# Detailed residual analysis for best model
best_model_name = results_df.iloc[0]['Model']
best_predictions = predictions[best_model_name]

plot_residuals_analysis(y_test, best_predictions, model_name=best_model_name)

In [ ]:
# Feature importance for best model
if best_model_name in ['Random Forest', 'XGBoost', 'LightGBM', 'Gradient Boosting']:
    importance_df = model_manager.get_feature_importance(best_model_name, X_train.columns.tolist())
    
    plt.figure(figsize=(12, 8))
    top_features = importance_df.head(15)
    sns.barplot(data=top_features, y='feature', x='importance', palette='viridis')
    plt.xlabel('Importance', fontsize=12)
    plt.ylabel('Feature', fontsize=12)
    plt.title(f'Top 15 Feature Importance - {best_model_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Top 10 Most Important Features:")
    print(importance_df.head(10).to_string(index=False))

## 7. Save Best Model

In [ ]:
# Save the best model and preprocessing artifacts
import joblib
import os

models_dir = '../models/saved_models'
os.makedirs(models_dir, exist_ok=True)

# Save best model
best_model = model_manager.trained_models[best_model_name]
joblib.dump(best_model, f'{models_dir}/best_model.pkl')
print(f"✅ Best model ({best_model_name}) saved!")

# Save scaler
joblib.dump(processed_data['scaler'], f'{models_dir}/scaler.pkl')
print("✅ Scaler saved!")

# Save feature names
joblib.dump(X_train.columns.tolist(), f'{models_dir}/feature_names.pkl')
print("✅ Feature names saved!")

# Save results
results_df.to_csv(f'{models_dir}/model_comparison.csv', index=False)
print("✅ Model comparison saved!")

print("\n🎉 All artifacts saved successfully!")

## 8. Sample Predictions

In [ ]:
# Make sample predictions
sample_indices = np.random.choice(X_test.index, 5, replace=False)
sample_X = X_test.loc[sample_indices]
sample_y_true = y_test.loc[sample_indices]
sample_y_pred = best_model.predict(sample_X)

print("\n🏠 Sample Predictions:")
print("="*60)
for i, (true_val, pred_val) in enumerate(zip(sample_y_true, sample_y_pred), 1):
    error = abs(true_val - pred_val)
    error_pct = (error / true_val) * 100
    print(f"Sample {i}:")
    print(f"  Actual:    ${true_val * 100000:,.0f}")
    print(f"  Predicted: ${pred_val * 100000:,.0f}")
    print(f"  Error:     ${error * 100000:,.0f} ({error_pct:.2f}%)")
    print("-"*60)

## 🎉 Project Complete!

### Summary:
- ✅ Data loaded and explored
- ✅ Advanced features engineered
- ✅ Multiple models trained and evaluated
- ✅ Best model selected and saved
- ✅ Ready for deployment!

### Next Steps:
1. Run the Streamlit dashboard: `streamlit run app/streamlit_app.py`
2. Make predictions on new data
3. Deploy to production